# Model judge: larger count-only test
800 claims (200 per building-count bucket, half correct, half incorrect) built from the xBD evidence counts. The truth follows from the bin definitions, so the rule-based checker is right by construction. This measures how often each judge gets the arithmetic wrong, by building count and quantity word. It does not test agreement with humans.

Setup: Runtime → Change runtime type → **T4 GPU**. Upload `judge_big.csv` when asked. Expect roughly 15 to 25 minutes per model. Results are saved to Google Drive as each model finishes.

## 1. Install and start Ollama

In [ ]:
import subprocess, shutil, time, requests

def ollama_up():
    try:
        requests.get("http://localhost:11434", timeout=3)
        return True
    except requests.exceptions.RequestException:
        return False

def install_ollama():
    if shutil.which("ollama") is None:
        subprocess.run("apt-get -qq install -y zstd && curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, check=True)

def start_ollama(wait=60):
    """Start the server if it is not running. Returns True when it answers."""
    if ollama_up():
        return True
    log = open("ollama.log", "w")
    subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log)
    for _ in range(wait // 2):
        time.sleep(2)
        if ollama_up():
            return True
    return False

install_ollama()
ok = start_ollama()
print("ollama up:", ok)
if not ok:
    print(open("ollama.log").read()[-1500:])

In [ ]:
MODELS = ['qwen3:8b', 'gemma3:12b', 'phi4:14b']
for m in MODELS:
    !ollama pull {m}

# Results are also saved to Google Drive, so a Colab disconnect does not lose them
import os
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/floodverify_judge/'
os.makedirs(SAVE, exist_ok=True)

## 2. Upload the data

In [ ]:
from google.colab import files
files.upload()  # choose judge_big.csv

import pandas as pd
df = pd.read_csv('judge_big.csv')
df['quantity'] = df.claim.str.split(' ').str[0]
print(df.shape)
print(df.truth.value_counts())
df.head()

## 3. Functions (same prompt as the first notebook)

In [ ]:
import re, time, requests

PROMPT = """You check one claim about building damage in a satellite tile.
Evidence (counts of buildings with usable imagery):
total={n_total}, undamaged={n_none}, minor={n_minor}, major={n_major}, destroyed={n_destroyed}.
Quantity words: none = 0%, few = under 10%, some = 10-30%, many = 30-60%, most = over 60%.
'damaged' means minor + major + destroyed.
Claim: "{claim}"
Answer with JSON only: {{"label": "correct" | "incorrect" | "cannot_tell"}}"""

URL = "http://localhost:11434/api/generate"

def build_prompt(row):
    return PROMPT.format(**row.to_dict())

def call_judge(prompt, model, tries=3, timeout=900):
    for k in range(tries):
        try:
            r = requests.post(URL, json={"model": model, "prompt": prompt, "stream": False,
                                         "think": False, "keep_alive": "30m",
                                         "options": {"temperature": 0, "seed": 0}}, timeout=timeout)
            text = r.json().get("response", "")
            m = re.search(r'"label"\s*:\s*"(correct|incorrect|cannot_tell)"', text)
            return m.group(1) if m else "cannot_tell"
        except requests.exceptions.ReadTimeout:
            time.sleep(5)
    return "cannot_tell"

def warm_up(model):
    """Load the model into memory before the first claim, so the first call does not time out."""
    requests.post(URL, json={"model": model, "prompt": "ok", "stream": False,
                             "keep_alive": "30m"}, timeout=1200)

def errors_by(res, col):
    t = res.assign(
        fc=(res.judge == "correct") & (res.truth == "incorrect"),
        fr=(res.judge == "incorrect") & (res.truth == "correct"))
    return t.groupby(col)[["fc", "fr"]].sum().rename(columns={"fc": "false_confirm", "fr": "false_reject"})

def plot_errors_by_bucket(results, order):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 3.5), sharey=True, squeeze=False)
    for ax, (m, r) in zip(axes[0], results.items()):
        errors_by(r, "bucket").loc[order].plot.bar(ax=ax, rot=0)
        ax.set_title(m)
    plt.tight_layout()

def run_judge(frame, model):
    out = frame.copy()
    out["judge"] = [call_judge(build_prompt(r), model) for _, r in frame.iterrows()]
    return out

def accuracy_by(res, col):
    t = res.assign(ok=(res.judge == res.truth))
    return t.groupby(col).ok.agg(["mean", "sum", "count"]).round(3)

def overall(res):
    ok = res.judge == res.truth
    return {"n": len(res), "accuracy": round(float(ok.mean()), 3),
            "false_confirmations": int(((res.judge == "correct") & (res.truth == "incorrect")).sum()),
            "false_rejections": int(((res.judge == "incorrect") & (res.truth == "correct")).sum()),
            "cannot_tell": int((res.judge == "cannot_tell").sum())}

def plot_by_bucket(results):
    import matplotlib.pyplot as plt
    order = ["1-3", "4-10", "11-30", "31+"]
    t = pd.DataFrame({m: accuracy_by(r, "bucket")["mean"] for m, r in results.items()}).loc[order]
    ax = t.plot(kind="bar", ylim=(0, 1), title="Judge accuracy by building count in the cell", rot=0)
    ax.set_ylabel("accuracy (checker = 1.0 by construction)")
    plt.tight_layout(); plt.show()

def plot_by_quantity(results):
    import matplotlib.pyplot as plt
    order = ["none", "few", "some", "many", "most"]
    t = pd.DataFrame({m: accuracy_by(r, "quantity")["mean"] for m, r in results.items()}).reindex(order)
    ax = t.plot(kind="bar", ylim=(0, 1), title="Judge accuracy by quantity word", rot=0)
    plt.tight_layout(); plt.show()

## 4. Smoke test on 4 rows

In [ ]:
print(run_judge(df.sample(4, random_state=0), 'qwen3:8b')[['claim','n_total','truth','judge']])

## 5. Run all models
Saves a CSV per model as it goes.

In [ ]:
results = {}
for m in MODELS:
    warm_up(m)
    results[m] = run_judge(df, m)
    name = f"judge_big_{m.replace(':', '_')}.csv"
    results[m].to_csv(name, index=False)
    results[m].to_csv(SAVE + name, index=False)
    print(m, overall(results[m]))

### 5b. Re-run only what is missing
If the runtime disconnected, reload the finished models from the local files or Google Drive and run only the missing ones. This cell also restarts the Ollama server if it is down. Re-upload `judge_big.csv` first if the runtime was reset. Any `cannot_tell` above zero means some requests still timed out, so run that model again.

In [ ]:
def load_saved(m):
    name = f"judge_big_{m.replace(':', '_')}.csv"
    for folder in ("", SAVE):
        if os.path.exists(folder + name):
            return pd.read_csv(folder + name)
    return None

start_ollama()
results = {m: r for m in MODELS if (r := load_saved(m)) is not None}
missing = [m for m in MODELS if m not in results]
print('missing:', missing)
for m in missing:
    warm_up(m)
    results[m] = run_judge(df, m)
    name = f"judge_big_{m.replace(':', '_')}.csv"
    results[m].to_csv(name, index=False)
    results[m].to_csv(SAVE + name, index=False)
    print(m, overall(results[m]))

## 6. Results

In [ ]:
table = pd.DataFrame({m: overall(r) for m, r in results.items()}).T
display(table)
for m, r in results.items():
    print(m, 'by building-count bucket')
    display(accuracy_by(r, 'bucket').loc[['1-3', '4-10', '11-30', '31+']])
plot_by_bucket(results)
plot_by_quantity(results)

In [ ]:
import pandas as pd

MODELS = ['qwen3:8b', 'gemma3:12b', 'phi4:14b']
results = {m: pd.read_csv(SAVE + f"judge_big_{m.replace(':', '_')}.csv") for m in MODELS}
for r in results.values():
    r['quantity'] = r.claim.str.split(' ').str[0]

def errors_by(res, col):
    t = res.assign(
        fc=(res.judge == "correct") & (res.truth == "incorrect"),
        fr=(res.judge == "incorrect") & (res.truth == "correct"))
    out = t.groupby(col)[["fc", "fr"]].sum().rename(columns={"fc": "false_confirm", "fr": "false_reject"})
    out["accuracy"] = (1 - (out.false_confirm + out.false_reject) / t.groupby(col).size()).round(3)
    out["n"] = t.groupby(col).size()
    return out

order = ["none", "few", "some", "many", "most"]
for m, r in results.items():
    print(m)
    display(errors_by(r, "quantity").reindex(order))

In [ ]:
def quantity_errors(results, order=("none", "few", "some", "many", "most")):
    rows = {m: errors_by(r, "quantity").reindex(order) for m, r in results.items()}
    return pd.concat(rows, axis=1)

display(quantity_errors(results))

### Where the errors sit
Accuracy and both error types by building-count bucket. If the false confirmations (and false rejections) pile up in the 1-3 and 4-10 buckets, the judges are mishandling small counts.

In [ ]:
order = ["1-3", "4-10", "11-30", "31+"]
for m, r in results.items():
    print(m)
    display(pd.concat([accuracy_by(r, "bucket"), errors_by(r, "bucket")], axis=1).loc[order])

In [ ]:
plot_errors_by_bucket(results, order)

## 7. Same table, correct vs incorrect claims separately
False confirmations (judge says correct, truth incorrect) are the cost that matters for verification.

In [ ]:
for m, r in results.items():
    print(m)
    display(pd.crosstab(r.truth, r.judge))